# Visualising interactions

Example of visualising interaction strength:

<video src="../figures/interaction-viz-example.webm" controls>

## Setup runner & utilities

The usual setup:

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../systems/openmm/17-ala.openmm.zip")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: interaction visuals")
imd_runner.load(0)

In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

Simplify rendering to reduce visual noise:

In [3]:
utilities.selections.update_selection("root", renderer=dict(color="pink", render="liquorice"))

## Interaction visuals

In [4]:
import math
import numpy as np

from nanover.imd.imd_force import calculate_imd_force, get_center_of_mass_subset
from nanover.trajectory import FrameData
from nanover.jupyter import FrameListener
from nanover.utilities.transforms import look_matrix

COLOR = [1, .25, .25, 1]

# precompute circle points
CIRCLE_POINTS = 16
CIRCLE_ANGLES = [
    math.pi * 2 * i / (CIRCLE_POINTS - 1) for i in range(CIRCLE_POINTS)
]
CIRCLE = np.array(
    [[math.cos(angle) / 2, math.sin(angle) / 2, 0] for angle in CIRCLE_ANGLES]
)


class InteractionVisuals(FrameListener):
    def on_frame_update(self, full_frame: FrameData, frame_update: FrameData):
        utilities.objects.clear()

        for key, interaction in imd_runner.app_server.imd.active_interactions.items():
            # compute energy and forces applied by this interaction
            imf = simulation.imd_force_manager
            center = get_center_of_mass_subset(
                full_frame.particle_positions,
                imf.masses,
                interaction.particles,
                imf.periodic_box_lengths,
            )
            energy, forces = calculate_imd_force(
                full_frame.particle_positions,
                imf.masses,
                [interaction],
                imf.periodic_box_lengths,
            )

            # compute total force applied
            force_mags = np.linalg.norm(forces[interaction.particles], axis=1)
            force_total = float(np.sum(force_mags))

            # circle looking along interaction from midpoint, scaled according to force
            radius = force_total / 1000
            midpoint = np.add(interaction.position, center) * .5
            rotation = look_matrix(np.subtract(interaction.position, center))[:3, :3]
            circle = (CIRCLE @ rotation) * radius + midpoint
            utilities.objects.update_line(
                f"force.{key}",
                positions=circle,
                color=COLOR,
            )

            # total force applied displayed inside circle
            utilities.objects.update_label(
                f"force.{key}",
                position=midpoint,
                text=f"{force_total:.0f}kJ/mol/nm",
            )


# set visuals running
interaction_visuals = InteractionVisuals.from_runner(imd_runner)
interaction_visuals.start()